In [18]:
import pandas as pd
from IPython.display import display
import yfinance as yf

import os

In [19]:
def load_process_data(file, hr=2): #hr用于调整时区
    df = pd.read_csv(file)
    df = df[["date","open","high","low","close","volume"]]

    df["date"] = pd.to_datetime(df["date"])
    df.columns = ["Datetime","Open","High","Low","Close","Volume"]
    df = df.set_index("Datetime")
    df.index = df.index + pd.Timedelta(hours=hr) #调整时区
    df = df.sort_index()
    df = df.between_time("09:30", "15:59")
    df = df[~df.index.duplicated(keep="first")]
    
    return df 

def load_process_yf_data():
    df = yf.download(tickers="^GSPC", period="8d", interval="1m")
    df.columns = df.columns.droplevel(1)
    df = df.reset_index()
    df = df[["Datetime","Close","High","Low","Open","Volume"]]
    df = df.rename(columns = {"Price":"Index"})
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.set_index("Datetime")
    df = df[~df.index.duplicated(keep="first")]
    df = df.sort_index()

    df.index = df.index + pd.Timedelta(hours=-5)

    return df



In [20]:
#处理SPY数据
df_raw = load_process_data("raw_data/1_min_SPY_2008-2021.csv")
display(df_raw.head())
df_raw.to_csv("processed_data/prc_1_min_SPY_2008-2021.csv", index=True, encoding="utf-8-sig")

,Open,High,Low,Close,Volume
Datetime,,,,,
2008-01-22 09:30:00,126.45,126.82,126.00,126.67,30987
2008-01-22 09:31:00,126.67,127.17,126.39,127.12,20111
2008-01-22 09:32:00,127.10,127.13,126.71,126.78,11979
2008-01-22 09:33:00,126.76,126.90,126.53,126.54,8017
2008-01-22 09:34:00,126.54,127.18,126.54,126.78,11967


处理跨资产数据

In [21]:
# 识别raw_data文件夹下的所有以multimkt开头的csv文件
import os

files = sorted([f for f in os.listdir("raw_data")
                if f.startswith("multimkt") and f.endswith(".csv")])

dfs = []

for file in files:
    path = os.path.join("raw_data", file)
    df = pd.read_csv(path)

    # 基础清洗
    df["date"] = pd.to_datetime(df["date"])

    dfs.append(df)

# 合并
df_multimkt = pd.concat(dfs, ignore_index=True)

# 排序（非常重要）
df_multimkt = df_multimkt.sort_values(["sym_root", "date"])

# 去重
df_multimkt = df_multimkt.drop_duplicates(subset=["sym_root", "date"], keep="first")

# 查看 symbols
sym = df_multimkt["sym_root"].unique()

In [23]:
for i in sym:
    df = df_multimkt[df_multimkt["sym_root"] == i].copy()

    df["Datetime"] = pd.to_datetime(
        df["date"].dt.strftime("%Y-%m-%d") + " " + df["minute"].astype(str)
    )

    df.drop(columns=["date","minute","sym_root","amount"], inplace=True)
    df.rename(columns={
        "open_price":"Open",
        "high_price":"High",
        "low_price":"Low",
        "close_price":"Close",
        "volume":"Volume"
    }, inplace=True)

    df = df.sort_values("Datetime")
    df = df.drop_duplicates(subset=["Datetime"], keep="first")
    df = df.set_index("Datetime")
    df = df.between_time("09:30", "15:59")

    df.to_csv(f"processed_data/prc_1_min_{i}_2012-2021.csv", encoding="utf-8-sig")

处理跨市场数据（恒生）

In [25]:
import os

files = sorted([f for f in os.listdir("raw_data")
                if f.startswith("RESSET") and f.endswith(".xlsx")])

dfs = []

for file in files:
    path = os.path.join("raw_data", file)
    df = pd.read_excel(path)

    if any(y in file for y in ["2019", "2020", "2021"]):
        df.columns = df.columns.str.split('_', n=1).str[-1]

    dfs.append(df)

df_concat = pd.concat(dfs, ignore_index=True)

# 构建 Datetime
df_concat["Datetime"] = pd.to_datetime(
    df_concat["Qdate"].astype(str) + " " + df_concat["StdTime"].astype(str)
)

# 只保留需要列
df = df_concat[["Datetime", "Begpr", "Highpr", "Lowpr", "TVolume_accu1"]].copy()

# 重命名
df = df.rename(columns={
    "Begpr": "Open",
    "Highpr": "High",
    "Lowpr": "Low",
    "TVolume_accu1": "Volume"
})

# 排序 + 去重
df = df.sort_values("Datetime")
df = df.drop_duplicates(subset=["Datetime"], keep="first")

display(df.head())

,Datetime,Open,High,Low,Volume
10240,2012-10-22 09:16:00,NaN,NaN,NaN,0
10291,2012-10-22 09:17:00,NaN,NaN,NaN,0
10294,2012-10-22 09:18:00,NaN,NaN,NaN,0
10299,2012-10-22 09:19:00,NaN,NaN,NaN,0
10305,2012-10-22 09:20:00,NaN,NaN,NaN,0


In [26]:
df.head(20)

,Datetime,Open,High,Low,Volume
10240,2012-10-22 09:16:00,NaN,NaN,NaN,0
10291,2012-10-22 09:17:00,NaN,NaN,NaN,0
10294,2012-10-22 09:18:00,NaN,NaN,NaN,0
10299,2012-10-22 09:19:00,NaN,NaN,NaN,0
10305,2012-10-22 09:20:00,NaN,NaN,NaN,0
10307,2012-10-22 09:21:00,NaN,NaN,NaN,0
10312,2012-10-22 09:22:00,NaN,NaN,NaN,0
10322,2012-10-22 09:23:00,NaN,NaN,NaN,0
10329,2012-10-22 09:24:00,NaN,NaN,NaN,0
10388,2012-10-22 09:25:00,NaN,NaN,NaN,0


In [27]:
# 导出 csv
df = df_concat[['Qdate', 'StdTime', 'Highpr', 'Lowpr', 'Begpr', 'TVolume_accu1']].copy()

# 构建时间
df["Datetime"] = pd.to_datetime(
    df["Qdate"].astype(str) + " " + df["StdTime"].astype(str)
)

# 删原列
df.drop(columns=["Qdate", "StdTime"], inplace=True)

# 重命名
df.rename(columns={
    "Begpr": "Open",
    "Highpr": "High",
    "Lowpr": "Low"
}, inplace=True)

# 设为时间索引并排序
df = df.set_index("Datetime")
df = df.sort_index()

# 只保留交易时段
df = df.between_time("09:30", "15:01")

# 如果 TVolume_accu1 是累计成交量，转成分钟成交量
df["Volume"] = df.groupby(df.index.date)["TVolume_accu1"].diff()
df["Volume"] = df["Volume"].fillna(df["TVolume_accu1"])
df.drop(columns=["TVolume_accu1"], inplace=True)

# 时间标签前移一分钟
df.index = df.index - pd.Timedelta(minutes=1)

# 用下一分钟 Open 构造当前分钟 Close
df = df.groupby(df.index.date, group_keys=False).apply(
    lambda x: x.assign(Close=x["Open"].shift(-1).fillna(x["Open"].iat[-1]))
)

# 调整列顺序
df = df[["Open", "High", "Low", "Close", "Volume"]]

# 导出前恢复普通列
df = df.reset_index()

display(df.head())

df.to_csv("processed_data/prc_1_min_HK_2012-2021.csv", index=False, encoding="utf-8-sig")

,Datetime,Open,High,Low,Close,Volume
0,2012-10-22 09:30:00,0.981,0.984,0.979,0.982,13806232.0
1,2012-10-22 09:31:00,0.982,0.987,0.982,0.985,-8474994.0
2,2012-10-22 09:32:00,0.985,0.986,0.984,0.985,-240848.0
3,2012-10-22 09:33:00,0.985,0.987,0.985,0.987,165572.0
4,2012-10-22 09:34:00,0.987,0.989,0.987,0.987,1435791.0


In [28]:
display(df.tail())

,Datetime,Open,High,Low,Close,Volume
563518,2021-12-31 14:56:00,1.213,1.213,1.212,1.213,1113000.0
563519,2021-12-31 14:57:00,1.213,1.213,1.213,1.213,-1715800.0
563520,2021-12-31 14:58:00,1.213,1.213,1.213,1.213,0.0
563521,2021-12-31 14:59:00,1.213,1.213,1.213,1.212,0.0
563522,2021-12-31 15:00:00,1.212,1.212,1.212,1.212,2151300.0
